# 6장 2강: 허깅페이스 모델 실습
## 2. 트랜스포머 아키텍처별 모델 실습


### 2.2 인코더 모델 활용: KoBERT로 감정 분류

In [3]:
# 토크나이저 및 모델 로드
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# KoBERT 토크나이저와 모델 로드
tokenizer = AutoTokenizer.from_pretrained("monologg/kobert", trust_remote_code=True)
model = AutoModelForSequenceClassification.from_pretrained("rkdaldus/ko-sent5-classification")

# 사용자 입력 텍스트 감정 분석
#text = "오늘 정말 행복해!"
text = "이번 시험을 위해 밤을 새우며 정말 열심히 준비했는데 생각보다 점수가 너무 낮게 나와서 속상하고 눈물이 나네."
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
print("inputs:", inputs)
with torch.no_grad():
    outputs = model(**inputs)
predicted_label = torch.argmax(outputs.logits, dim=1).item()

# 감정 레이블 정의
emotion_labels = {
    0: ("Angry", "😡"),
    1: ("Fear", "😨"),
    2: ("Happy", "😊"),
    3: ("Tender", "🥰"),
    4: ("Sad", "😢")
}

# 예측된 감정 출력
print(f"예측된 감정: {emotion_labels[predicted_label][0]} {emotion_labels[predicted_label][1]}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10882.08it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: rkdaldus/ko-sent5-classification
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


inputs: {'input_ids': tensor([[   2, 3697, 3005, 7088, 3567, 2265, 7088, 2695, 7005, 6197, 4102, 3367,
         4249, 7867, 2705, 6371, 4077, 5330, 1458, 1429, 5400, 1394, 6553, 2856,
         6527, 7788, 1537, 7096, 1370, 5702,   54,    3]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1]])}
예측된 감정: Happy 😊


### 2.3 인코더-디코더 모델 활용: KoBART로 뉴스 요약

In [4]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

text = "별아교세포를 성숙 뉴런으로 재프로그래밍하여 손상된 인지 기능 완벽 회복 유전자 변형 없는 단백질 표적 분해 기술로 안전성 확보 및 퇴행성 뇌 질환 치료 새 지평"

raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

summary_ids = model.generate(torch.tensor([input_ids]))
tokenizer.decode(summary_ids.squeeze().tolist(), skip_special_tokens=True)

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 10245.19it/s]
c:\Users\jihyu\Documents\AX_study\04.Machine Learning\.venv\Lib\site-packages\transformers\generation\utils.py:1676: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


'별아교세포를 성숙 뉴런으로 재프로그래밍하여 손상된 인지 기능 완벽 회복'

### 2.4 디코더 모델 활용: Gemma로 대화형 텍스트 생성

- 디코더 전용 구조는 문장 생성(NLG) 태스크를 수행합니다. 허깅페이스의 상위 인터페이스인 pipeline 구조 내부에서 가속기를 활용할 수 있도록 device_map="auto" 옵션을 주입합니다. 가상환경 내에 설치된 accelerate 패키지가 각 OS 운영체제 환경을 판별하여 최적의 가속 레이어(CUDA 또는 MPS)를 자동 매핑합니다.

In [5]:
# 교안 코딩 : 
from transformers import pipeline
import torch

# Gemma 인스트럭션 모델 식별자 설정
gemma_identifier = "google/gemma-2b-it"

# 텍스트 생성 파이프라인 구축 (bfloat16 연산 및 자원 자동 배치 적용)
gemma_generator = pipeline(
    "text-generation",
    model=gemma_identifier,
    dtype=torch.bfloat16,
    device_map="auto" # accelerate를 통해 OS별 하드웨어(CUDA/MPS) 최적 자동 할당
)

# 대화형 프롬프트 구조 정의
user_dialogue = [
    {"role": "user", "content": "클라우드 컴퓨팅의 개념을  딱 한 문장으로 재미있게 비유해서 알려줘."}
]

# 답변 생성
outputs = gemma_generator(user_dialogue, max_new_tokens=200)
print(outputs)


# 최종 답변 추출 및 출력
ai_reply = outputs[0]["generated_text"][-1]["content"]
print("--- Gemma 답변 ---")
print(f"Gemma의 답변:\n{ai_reply}")

Loading weights: 100%|██████████| 164/164 [00:00<00:00, 3667.78it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cle

[{'generated_text': [{'role': 'user', 'content': '클라우드 컴퓨팅의 개념을  딱 한 문장으로 재미있게 비유해서 알려줘.'}, {'role': 'assistant', 'content': '클라우드 컴퓨팅은 멀티플레이스 컴퓨팅과 클라우드 컴퓨팅의 중간적인 개념으로, 컴퓨터 기반 시스템의 일부 부분을 중앙 서버에서 분산시키는 것을 의미합니다.'}]}]
--- Gemma 답변 ---
Gemma의 답변:
클라우드 컴퓨팅은 멀티플레이스 컴퓨팅과 클라우드 컴퓨팅의 중간적인 개념으로, 컴퓨터 기반 시스템의 일부 부분을 중앙 서버에서 분산시키는 것을 의미합니다.
